In [7]:

%pip install streamlit joblib scikit-learn nltk


Note: you may need to restart the kernel to use updated packages.


In [8]:

%%writefile researchly_app.py

import streamlit as st
import joblib
import re
import nltk

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)

st.set_page_config(
    page_title="Researchly Research Abstract Classifier",
    page_icon="📚",
    layout="centered"
)

@st.cache_resource
def load_model():
    return joblib.load("researchly_best_model.joblib")

model = load_model()

stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'[^a-z\s]', ' ', text)

    tokens = word_tokenize(text)

    tokens = [
        stemmer.stem(word)
        for word in tokens
        if word not in stop_words and len(word) > 2
    ]

    return ' '.join(tokens)

st.title("Researchly Research Abstract Classifier")

st.write(
    "This app classifies research abstracts as either "
    "AI-related or classical research."
)

st.divider()

st.subheader("Enter Research Abstract")

abstract = st.text_area(
    "Enter research abstract:",
    height=250,
    placeholder="Type or paste your research abstract here..."
)

if st.button("Classify Abstract"):

    if abstract.strip() == "":
        st.warning("Please enter a research abstract.")

    else:
        cleaned_abstract = clean_text(abstract)

        if not cleaned_abstract:
            st.warning("Please enter a valid research abstract.")

        else:
            prediction = model.predict([cleaned_abstract])[0]

            st.subheader("Prediction")

            if str(prediction).lower() == "ai":
                st.success(
                    "The abstract is classified as: "
                    "Artificial Intelligence"
                )

            elif str(prediction).lower() == "classical":
                st.success(
                    "The abstract is classified as: "
                    "Classical Research"
                )

            else:
                st.info(
                    f"The abstract is classified as: {prediction}"
                )


Writing researchly_app.py


In [12]:

import subprocess
import sys
import os
import time

print("App file exists:", os.path.exists("researchly_app.py"))
print("Model file exists:", os.path.exists("researchly_best_model.joblib"))

process = subprocess.Popen(
    [
        sys.executable,
        "-m",
        "streamlit",
        "run",
        "researchly_app.py",
        "--server.port=8501",
        "--server.headless=true"
    ],
    stdout=open("streamlit.log", "w"),
    stderr=subprocess.STDOUT
)

time.sleep(5)

print("Process status:", process.poll())

with open("streamlit.log", "r") as file:
    print(file.read())



App file exists: True
Model file exists: True
Process status: None



  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://192.168.10.15:8501
  External URL: http://105.244.162.148:8501




Process status: Process variable not saved
